<a href="https://colab.research.google.com/github/stpaul2coderdojo/Pair-with-Github-Copilot/blob/main/cmLawnMowerCu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install cupy-cuda12x  # For CUDA 12 (latest)


In [2]:
import cupy as cp

# Example data initialization using CuPy arrays
Number_of_Lawns_mowed = cp.array([2, 1, 4, 1, 3], dtype=cp.int32)
Average_Size_of_Lawns = cp.array([500, 600, 550, 400, 700], dtype=cp.int32)
Total_Time = cp.array([60, 80, 90, 50, 120], dtype=cp.float32)
Type_of_Lawn_Mower = cp.array([1, 2, 1, 3, 2], dtype=cp.int32)

num_candidates = Number_of_Lawns_mowed.size

# GPU Kernel code
kernel_code = """
extern "C" __global__ void Fastest_Lawn_Mower(int* num_mowed, int* avg_size, float* total_time,
                                               int* mower_type, int* fastest_workers, int* fastest_mowers, float* fastest_speeds) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    if (idx >= %(num_candidates)s) return;

    int area = num_mowed[idx] * avg_size[idx];
    float speed = area / total_time[idx];

    if (fastest_workers[area] == -1 || fastest_speeds[area] < speed) {
        fastest_mowers[area] = mower_type[idx];
        fastest_workers[area] = idx;
        fastest_speeds[area] = speed;
    }
}
"""

# Compiling kernel
module = cp.RawModule(code=kernel_code % {"num_candidates": num_candidates})
Fastest_Lawn_Mower = module.get_function("Fastest_Lawn_Mower")

# Allocate output arrays on GPU
fastest_workers = cp.full(1001, -1, dtype=cp.int32)
fastest_mowers = cp.full(1001, -1, dtype=cp.int32)
fastest_speeds = cp.full(1001, -1.0, dtype=cp.float32)

# Launch kernel
grid_size = (num_candidates + 31) // 32
block_size = 32
Fastest_Lawn_Mower((grid_size,), (block_size,), (Number_of_Lawns_mowed, Average_Size_of_Lawns, Total_Time, Type_of_Lawn_Mower,
                                                 fastest_workers, fastest_mowers, fastest_speeds))

# Fetch results
print("Fastest Workers:", fastest_workers.get())
print("Fastest Mowers:", fastest_mowers.get())
print("Fastest Speeds:", fastest_speeds.get())


Fastest Workers: [-1 -1 -1 ... -1 -1  0]
Fastest Mowers: [-1 -1 -1 ... -1 -1  1]
Fastest Speeds: [-1.       -1.       -1.       ... -1.       -1.       16.666666]
